In [ ]:
# Celda 1
# Paths and parameters:
%cd ..

GPKG = "data/catastro_manzana.gpkg"
CLOUD = "data/nube_manzana_2023.laz"
JSON_FILE = "data/lod2_meshes.json"
BLEND_FILE = "blender/lod2.blend"
GLB_FILE = "blender/lod2.glb"
BLENDER = "C:/Program Files/Blender Foundation/Blender 5.2/blender.exe"
BPY_SCRIPT = "scripts/meshes_to_blender.py"

# Local origin of the Blender scene, in EPSG:25830 meters (same as LOD1):
ORIGIN = (440000.0, 4473700.0, 640.0)

# Parcels of the Colegio Imperial get a LOD2 roof; the rest of the block stays LOD1:
CONJUNTO = ["0139402VK4703G", "0139423VK4703G", "0139424VK4703G"]

BUILDING_CLASS = 6
CELL = 0.5          # Grid size of the roof surface, in meters
PERCENTILE = 90     # Roof height of a grid node = this percentile of the points around it
SMOOTH_PASSES = 1   # Times the 3 x 3 median filter is applied to the roof grid (0 = no smoothing)

# Roof pitches: planes found in the grid of each part (RANSAC); every node near a plane takes its height,
# so that the pitches come out flat and the noise steps go, while the real steps stay. See Celda 7.
PLANE_SNAP = True
PLANE_TOL = 0.2      # A node fits a plane if it is within this many meters
PLANE_MIN = 30       # Smallest plane, in nodes (30 x 0.25 m2 = 7.5 m2)
PLANE_ITER = 800     # Random tries per plane
PLANE_SEARCH = 6     # The 3 nodes of a try are within this many cells of each other
PLANE_MAX_SLOPE = 1.5   # Steeper than this (56 degrees) is not a roof pitch
SNAP_TOL = 0.45      # A node takes the height of a plane if it is within this many meters...
SNAP_REACH = 3       # ...and within this many cells of the nodes that defined the plane
SNAP_TOL_2 = 1.5     # Second pass, for the nodes left (noise spikes, chimneys)
MAJORITY = 5         # Window (cells) of the majority filter that straightens the borders between pitches
PLANE_SEED = 0

In [ ]:
# Celda 2
import os
import json
import subprocess
import numpy as np
from scipy import ndimage
import laspy
import shapely
import geopandas as gpd
from shapely.geometry.polygon import orient

In [ ]:
# Celda 3
# Building parts with heights (from 03_alturas) and building points of the cloud:
parts = gpd.read_file(GPKG, layer="building_parts_alturas")
parts = parts[parts["numberOfFloorsAboveGround"] > 0].copy()
parts["z_top"] = parts["z_base"] + parts["height_m"]

cloud = laspy.read(CLOUD)
is_building = np.array(cloud.classification) == BUILDING_CLASS
bx = np.array(cloud.x)[is_building]
by = np.array(cloud.y)[is_building]
bz = np.array(cloud.z)[is_building]
print(len(parts), "parts |", len(bz), "building points")

In [ ]:
# Celda 4
# Roof grid of one part: a height for every node of a grid of CELL meters.
# Each node takes the PERCENTILE of the building points (inside the footprint) that are closer to it than to any other node.
# Returns the grid corner (gx0, gy0) and the array of heights Z[row, column] (NaN where there are no points).
def roof_grid(polygon):
    xmin, ymin, xmax, ymax = polygon.bounds
    gx0 = np.floor(xmin / CELL) * CELL - CELL   # One extra node on each side
    gy0 = np.floor(ymin / CELL) * CELL - CELL
    n_cols = int(np.ceil((xmax - gx0) / CELL)) + 2
    n_rows = int(np.ceil((ymax - gy0) / CELL)) + 2
    Z = np.full((n_rows, n_cols), np.nan)

    near = (bx >= xmin) & (bx <= xmax) & (by >= ymin) & (by <= ymax)
    px = bx[near]
    py = by[near]
    pz = bz[near]
    inside = shapely.contains_xy(polygon, px, py)
    px = px[inside]
    py = py[inside]
    pz = pz[inside]

    # Nearest node of each point: (x = 440012.3, gx0 = 440000, CELL = 0.5) -> column round(24.6) = 25
    cols = np.round((px - gx0) / CELL).astype(int)
    rows = np.round((py - gy0) / CELL).astype(int)
    keys = rows * n_cols + cols
    order = np.argsort(keys)
    keys = keys[order]
    pz = pz[order]
    unique_keys, starts = np.unique(keys, return_index=True)
    groups = np.split(pz, starts[1:])
    for key, values in zip(unique_keys, groups):
        Z[key // n_cols, key % n_cols] = np.percentile(values, PERCENTILE)
    return gx0, gy0, Z

In [ ]:
# Celda 5
# Fill the nodes without points with the mean of their filled neighbors (repeat until none is left):
def fill_grid(Z, fallback):
    Z = Z.copy()
    if np.isnan(Z).all():
        Z[:, :] = fallback
        return Z
    n_rows, n_cols = Z.shape
    while np.isnan(Z).any():
        new_Z = Z.copy()
        for r in range(n_rows):
            for c in range(n_cols):
                if not np.isnan(Z[r, c]):
                    continue
                neighbors = Z[max(r - 1, 0):r + 2, max(c - 1, 0):c + 2]
                if not np.isnan(neighbors).all():
                    new_Z[r, c] = np.nanmean(neighbors)
        Z = new_Z
    return Z

In [ ]:
# Celda 6
# Smooth the grid: each node takes the median of itself and its 8 neighbors (3 x 3 window).
# This removes the noise of the cloud but keeps the edges of roofs (a median does not blur steps).
def median_grid(Z):
    n_rows, n_cols = Z.shape
    new_Z = Z.copy()
    for r in range(n_rows):
        for c in range(n_cols):
            window = Z[max(r - 1, 0):r + 2, max(c - 1, 0):c + 2]
            new_Z[r, c] = np.median(window)
    return new_Z

In [ ]:
# Celda 7
# Height of the roof at any point (x, y), by bilinear interpolation between the 4 nodes around it:
def roof_height(x, y, gx0, gy0, Z):
    fx = (x - gx0) / CELL
    fy = (y - gy0) / CELL
    c = int(np.floor(fx))
    r = int(np.floor(fy))
    tx = fx - c
    ty = fy - r
    z_low = Z[r, c] * (1 - tx) + Z[r, c + 1] * tx
    z_high = Z[r + 1, c] * (1 - tx) + Z[r + 1, c + 1] * tx
    return z_low * (1 - ty) + z_high * ty

In [ ]:
# Celda 7b
# Roof pitches as planes. In the grid of a part (raw, before filling), find planes one by one: try PLANE_ITER planes
# through 3 nearby nodes, keep the one that more nodes fit (within PLANE_TOL), refit it by least squares, and keep its
# largest connected patch. Repeat with the nodes left until 5 tries in a row find nothing.
# Then every node of the filled grid that is within SNAP_TOL of a plane, and near its patch, takes the plane height
# (the closest plane if several).
plane_rng = np.random.default_rng(PLANE_SEED)

def find_planes(gx0, gy0, Z):
    n_rows, n_cols = Z.shape
    R, C = np.mgrid[0:n_rows, 0:n_cols]
    X = gx0 + C * CELL
    Y = gy0 + R * CELL
    remaining = ~np.isnan(Z)
    planes = []
    failures = 0
    while remaining.sum() >= PLANE_MIN and failures < 5:
        nodes = np.argwhere(remaining)
        best = None
        for _ in range(PLANE_ITER):
            r0, c0 = nodes[plane_rng.integers(len(nodes))]
            near = nodes[(np.abs(nodes[:, 0] - r0) <= PLANE_SEARCH) & (np.abs(nodes[:, 1] - c0) <= PLANE_SEARCH)]
            if len(near) < 3:
                continue
            pick = near[plane_rng.choice(len(near), 3, replace=False)]
            A = np.c_[np.ones(3), X[pick[:, 0], pick[:, 1]], Y[pick[:, 0], pick[:, 1]]]
            if abs(np.linalg.det(A)) < 1e-6:
                continue
            coef = np.linalg.solve(A, Z[pick[:, 0], pick[:, 1]])   # z = a + b x + c y
            if np.hypot(coef[1], coef[2]) > PLANE_MAX_SLOPE:
                continue
            fit = remaining & (np.abs(Z - (coef[0] + coef[1] * X + coef[2] * Y)) < PLANE_TOL)
            if best is None or fit.sum() > best[0]:
                best = (fit.sum(), coef, fit)
        if best is None or best[0] < PLANE_MIN:
            failures += 1
            continue
        _, coef, fit = best
        for _ in range(2):
            A = np.c_[np.ones(fit.sum()), X[fit], Y[fit]]
            coef = np.linalg.lstsq(A, Z[fit], rcond=None)[0]
            fit = remaining & (np.abs(Z - (coef[0] + coef[1] * X + coef[2] * Y)) < PLANE_TOL)
        labels, count = ndimage.label(fit)
        if count == 0:
            failures += 1
            continue
        sizes = ndimage.sum(fit, labels, range(1, count + 1))
        patch = labels == (np.argmax(sizes) + 1)
        remaining &= ~patch
        if patch.sum() >= PLANE_MIN:
            planes.append((coef, patch))
            failures = 0
        else:
            failures += 1
    return planes

def snap_to_planes(gx0, gy0, Z, planes):
    n_rows, n_cols = Z.shape
    R, C = np.mgrid[0:n_rows, 0:n_cols]
    X = gx0 + C * CELL
    Y = gy0 + R * CELL
    heights = [coef[0] + coef[1] * X + coef[2] * Y for coef, patch in planes]
    reaches = [ndimage.binary_dilation(patch, iterations=SNAP_REACH) for coef, patch in planes]

    # Label of each node: the closest plane within SNAP_TOL (first pass), or within SNAP_TOL_2 for the nodes left
    # (second pass: noise spikes and small things on the roof go to the pitch around them). -1 = no plane.
    label = np.full(Z.shape, -1)
    for tolerance in (SNAP_TOL, SNAP_TOL_2):
        closest = np.full(Z.shape, np.inf)
        for k, (plane, reach) in enumerate(zip(heights, reaches)):
            dz = np.abs(Z - plane)
            take = (label == -1) & reach & (dz < tolerance) & (dz < closest)
            closest[take] = dz[take]
        for k, (plane, reach) in enumerate(zip(heights, reaches)):
            dz = np.abs(Z - plane)
            label[(label == -1) & reach & (dz == closest)] = k

    # Straighter borders between pitches: each labeled node takes the most common label in its MAJORITY x MAJORITY
    # window (only among labeled nodes), if that plane is within SNAP_TOL_2 there.
    if planes:
        counts = np.stack([ndimage.uniform_filter((label == k).astype(float), size=MAJORITY) for k in range(len(planes))])
        majority = np.argmax(counts, axis=0)
        for k, plane in enumerate(heights):
            switch = (label >= 0) & (majority == k) & (np.abs(Z - plane) < SNAP_TOL_2)
            label[switch] = k

    Z = Z.copy()
    for k, plane in enumerate(heights):
        Z[label == k] = plane[label == k]
    return Z, label >= 0


In [ ]:
# Celda 8
# LOD2 mesh of one part, in local coordinates:
# The footprint is cut by the grid squares. For each piece:
#   roof: its triangles lifted to the roof surface;
#   bottom: the same triangles at z_base, facing down;
#   walls: its edges that lie on the footprint border, from z_base up to the roof.
# Roof and walls are built from the same pieces, so they share their vertices (no cracks).
def lod2_mesh(polygon, z_base, gx0, gy0, Z):
    x0, y0, z0 = ORIGIN
    border = polygon.boundary
    vertices = []
    faces = []

    n_rows, n_cols = Z.shape
    for r in range(n_rows - 1):
        for c in range(n_cols - 1):
            square = shapely.box(gx0 + c * CELL, gy0 + r * CELL, gx0 + (c + 1) * CELL, gy0 + (r + 1) * CELL)
            piece = polygon.intersection(square)
            for small in shapely.get_parts(piece):
                if small.geom_type != "Polygon" or small.area == 0:
                    continue
                small = orient(small, 1.0)   # Outer ring counterclockwise, holes clockwise

                # Roof and bottom:
                for triangle in shapely.constrained_delaunay_triangles(small).geoms:
                    triangle = orient(triangle, 1.0)
                    corners = list(triangle.exterior.coords)[:3]
                    start = len(vertices)
                    for x, y in corners:
                        vertices.append((x - x0, y - y0, roof_height(x, y, gx0, gy0, Z) - z0))
                    for x, y in corners:
                        vertices.append((x - x0, y - y0, z_base - z0))
                    faces.append((start, start + 1, start + 2))
                    faces.append((start + 5, start + 4, start + 3))

                # Walls (only the edges on the footprint border, not the cuts of the grid):
                rings = [small.exterior] + list(small.interiors)
                for ring in rings:
                    coords = list(ring.coords)
                    for k in range(len(coords) - 1):
                        ax, ay = coords[k]
                        bx_, by_ = coords[k + 1]
                        middle = shapely.Point((ax + bx_) / 2, (ay + by_) / 2)
                        if border.distance(middle) > 1e-6:
                            continue
                        start = len(vertices)
                        vertices.append((ax - x0, ay - y0, z_base - z0))
                        vertices.append((bx_ - x0, by_ - y0, z_base - z0))
                        vertices.append((bx_ - x0, by_ - y0, roof_height(bx_, by_, gx0, gy0, Z) - z0))
                        vertices.append((ax - x0, ay - y0, roof_height(ax, ay, gx0, gy0, Z) - z0))
                        faces.append((start, start + 1, start + 2, start + 3))

    return vertices, faces

In [ ]:
# Celda 9
# LOD1 prism (same as in 04_lod1), for the context parts:
def prism(polygon, z_bottom, z_top):
    polygon = orient(polygon, 1.0)
    x0, y0, z0 = ORIGIN
    vertices = []
    faces = []
    rings = [polygon.exterior] + list(polygon.interiors)
    for ring in rings:
        coords = list(ring.coords)[:-1]
        n = len(coords)
        start = len(vertices)
        for x, y in coords:
            vertices.append((x - x0, y - y0, z_bottom - z0))
            vertices.append((x - x0, y - y0, z_top - z0))
        for i in range(n):
            j = (i + 1) % n
            a_low = start + 2 * i
            b_low = start + 2 * j
            faces.append((a_low, b_low, b_low + 1, a_low + 1))
    for triangle in shapely.constrained_delaunay_triangles(polygon).geoms:
        triangle = orient(triangle, 1.0)
        corners = list(triangle.exterior.coords)[:3]
        start = len(vertices)
        for x, y in corners:
            vertices.append((x - x0, y - y0, z_top - z0))
        for x, y in corners:
            vertices.append((x - x0, y - y0, z_bottom - z0))
        faces.append((start, start + 1, start + 2))
        faces.append((start + 5, start + 4, start + 3))
    return vertices, faces

In [ ]:
# Celda 10
# Build all meshes: LOD2 for the conjunto (when the part has cloud points), LOD1 for the rest:
meshes = []
snap_stats = []
for i, row in parts.iterrows():
    name = row["parcel"] + "_" + str(row["localId"]).split("_")[-1]
    if row["parcel"] in CONJUNTO and row["height_method"] == "cloud_p90":
        gx0, gy0, Z_raw = roof_grid(row.geometry)
        Z = fill_grid(Z_raw, row["z_top"])
        for k in range(SMOOTH_PASSES):
            Z = median_grid(Z)
        if PLANE_SNAP:
            planes = find_planes(gx0, gy0, Z_raw)
            Z, snapped = snap_to_planes(gx0, gy0, Z, planes)
            roof_nodes = ~np.isnan(Z_raw)
            snap_stats.append((len(planes), int((snapped & roof_nodes).sum()), int(roof_nodes.sum())))
        vertices, faces = lod2_mesh(row.geometry, row["z_base"], gx0, gy0, Z)
        group = "conjunto"
        lod = 2
    else:
        vertices, faces = prism(row.geometry, row["z_base"], row["z_top"])
        if row["parcel"] in CONJUNTO:
            group = "conjunto"
        else:
            group = "contexto"
        lod = 1
    meshes.append({
        "name": name,
        "parcel": row["parcel"],
        "group": group,
        "lod": lod,
        "height_m": float(row["height_m"]),
        "height_method": row["height_method"],
        "vertices": vertices,
        "faces": faces,
    })

lod2_count = sum(1 for m in meshes if m["lod"] == 2)
print(len(meshes), "meshes |", lod2_count, "with LOD2 roof |", sum(len(m["faces"]) for m in meshes), "faces")
if PLANE_SNAP:
    print(sum(s[0] for s in snap_stats), "roof planes |", round(100 * sum(s[1] for s in snap_stats) / sum(s[2] for s in snap_stats)),
          "% of the roof nodes on a plane")


In [ ]:
# Celda 11
# Save the meshes and run Blender:
with open(JSON_FILE, "w", encoding="utf-8") as f:
    json.dump({"origin": ORIGIN, "parts": meshes}, f)

command = [BLENDER, "-b", "--factory-startup", "--python", BPY_SCRIPT, "--", JSON_FILE, BLEND_FILE, GLB_FILE]
result = subprocess.run(command, capture_output=True, text=True)
for line in result.stdout.splitlines():
    if "Objects" in line or "Error" in line or "Saved" in line:
        print(line)
print("Return code:", result.returncode)
if result.returncode != 0:
    print(result.stderr[-2000:])

In [ ]:
# Celda 12
# Check the output files:
for path in [JSON_FILE, BLEND_FILE, GLB_FILE]:
    print(path, round(os.path.getsize(path) / 1e6, 2), "MB")